# Reverse-engineering the ESS-DIVE CROCUS WXT product

*A worked provenance / QA-QC investigation.*

## Background for readers

CROCUS instruments stream high-frequency data to the Sage/Waggle platform (the raw
"Tier 1" record). A subset of that data was quality-controlled, downsampled, and
published to **ESS-DIVE** as a citable dataset (a "Tier 2" product). The ESS-DIVE
packages, however, **do not document the exact downsampling procedure** — they give the
output but not the recipe.

This notebook reconstructs that recipe empirically. The method is simple: pull the
*native* high-frequency data from Sage for a short window where an ESS-DIVE file also
exists, apply candidate aggregations (mean, decimation, block max, vector mean, ...),
and see which one reproduces the published ESS-DIVE values.

**Why bother?** For any derived dataset, knowing *how* it was built determines how far
you can trust it. A specific question motivated this: the published wind field includes
a `wind_mean_10s` variable — was that mean computed correctly as a **vector average**
(decomposing wind into u/v components before averaging, the meteorologically correct
method), or as a naive **scalar average of wind speed** (a common error that inflates
mean wind when direction varies)? We set out to check.

**Test case:** NEIU rooftop station, Sage VSN **W08D**, 2023-08-30, 12:00-12:10 UTC.

## Summary of findings (for the impatient)

| ESS-DIVE variable | how it is built | verified? |
|---|---|---|
| `temperature`  | 10-second block **mean** of the native stream | **yes** (~3e-4 °C) |
| `wind_max_10s` | 10-second block **max** of native `wxt.wind.speed` | **yes, exact** (0.0) |
| `wind_mean_10s` | *not reproducible* from the high-rate speed | **no** — see §7-9 |

Along the way we also found two **documentation discrepancies**: the native sampling
rate is ~12 Hz (not the 1 Hz the ESS-DIVE description states for the thermodynamic
channels), and the description advertises the data for turbulence studies that the
published 10-second product cannot actually support.


## 1. Setup and cadence check

Establish the *output* cadence we are trying to reproduce, by differencing consecutive
timestamps in an ESS-DIVE file.

**Result:** WXT files are on a **10-second** step.
The 10-second WXT cadence is stable across both the July and August 2023 files.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import sage_data_client

# Two WXT files from the ESS-DIVE downloads, both are 10 s.
ds  = xr.open_dataset("data/essdive/tmp/NEIU_wxt/crocus_neiu_wxt_a1_20230520_000000.nc")
ds2 = xr.open_dataset("data/essdive/tmp/NEIU_wxt/crocus_neiu_wxt_a1_20230720_000000.nc")

# Time step = difference between consecutive timestamps, in milliseconds.
dt  = np.diff(ds['time'].values[:20]).astype('timedelta64[ms]')
dt2 = np.diff(ds2['time'].values[:20]).astype('timedelta64[ms]')

print("WXT step (ms):", dt)    
print("WXT step (ms):", dt2)   # -> 10000 ms    = 10-second cadence


WXT step (ms): [10000 10000 10000 10000 10000 10000 10000 10000 10000 10000 10000 10000
 10000 10000 10000 10000 10000 10000 10000]
WXT step (ms): [10000 10000 10000 10000 10000 10000 10000 10000 10000 10000 10000 10000
 10000 10000 10000 10000 10000 10000 10000]


## 2. Pull the native high-frequency stream from Sage

Compare against a short window of *native* data — NEIU (VSN **W08D**), 2023-08-30, a
10-minute window (short, because the native stream is high rate).

**Documentation discrepancy #1:** the ESS-DIVE description says temperature/RH/pressure/
rainfall are sampled at **1 Hz**. The native stream returns ~7250 samples in 10 minutes
≈ **12 Hz** — the thermodynamic channels are *not* 1 Hz in the raw record.

In [24]:
df = sage_data_client.query(
    start="2023-08-30T12:00:00Z",
    end="2023-08-30T12:10:00Z",
    filter={"name": "wxt.env.temp", "vsn": "W08D"},
)
print(df.shape, "rows")   # ~7000+ rows in 10 min -> ~10-12 Hz, NOT 1 Hz
df.head()


(7257, 12) rows


,timestamp,name,value,meta.host,meta.missing,meta.node,meta.plugin,meta.sensor,meta.task,meta.units,meta.vsn,meta.zone
0,2023-08-30 12:00:00.069286218+00:00,wxt.env.temp,13.7,000048b02d3ae277.ws-nxcore,-9999.9,000048b02d3ae277,registry.sagecontinuum.org/jrobrien/waggle-wxt...,vaisala-wxt536,waggle-wxt536,degree Celsius,W08D,core
1,2023-08-30 12:00:00.148104158+00:00,wxt.env.temp,13.7,000048b02d3ae277.ws-nxcore,-9999.9,000048b02d3ae277,registry.sagecontinuum.org/jrobrien/waggle-wxt...,vaisala-wxt536,waggle-wxt536,degree Celsius,W08D,core
2,2023-08-30 12:00:00.242447532+00:00,wxt.env.temp,13.7,000048b02d3ae277.ws-nxcore,-9999.9,000048b02d3ae277,registry.sagecontinuum.org/jrobrien/waggle-wxt...,vaisala-wxt536,waggle-wxt536,degree Celsius,W08D,core
3,2023-08-30 12:00:00.322090761+00:00,wxt.env.temp,13.7,000048b02d3ae277.ws-nxcore,-9999.9,000048b02d3ae277,registry.sagecontinuum.org/jrobrien/waggle-wxt...,vaisala-wxt536,waggle-wxt536,degree Celsius,W08D,core
4,2023-08-30 12:00:00.402097803+00:00,wxt.env.temp,13.7,000048b02d3ae277.ws-nxcore,-9999.9,000048b02d3ae277,registry.sagecontinuum.org/jrobrien/waggle-wxt...,vaisala-wxt536,waggle-wxt536,degree Celsius,W08D,core


## 3. Build candidate downsamplings of the native temperature

Two hypotheses for producing a 10-second value: **block mean** (average all native
samples in each 10-second bin) or **decimation** (keep one sample per bin). Compute both.

In [25]:
nat = df[['timestamp', 'value']].copy()
nat['timestamp'] = pd.to_datetime(nat['timestamp'], utc=True)
nat = nat.set_index('timestamp')['value']

mean_10s  = nat.resample('10s').mean()    # candidate 1: block mean
first_10s = nat.resample('10s').first()   # candidate 2: decimation (first sample)


## 4. Inspect the ESS-DIVE WXT variables

The names are informative: `wind_dir_10s`, `wind_mean_10s`, `wind_max_10s` — the `_10s`
suffix and mean/max split show wind was **aggregated into 10-second statistics**,
implying block aggregation. `dewpoint`/`wetbulb` are *derived*, not raw channels.

In [26]:
essd = xr.open_dataset("data/essdive/tmp/NEIU_wxt/crocus_neiu_wxt_a1_20230830_000000.nc")
print("ESS-DIVE data_vars:", list(essd.data_vars))


ESS-DIVE data_vars: ['temperature', 'humidity', 'pressure', 'rainfall', 'dewpoint', 'wetbulb', 'wind_dir_10s', 'wind_mean_10s', 'wind_max_10s']


## 5. Temperature: block mean vs. decimation

Temperature is quantized and slowly varying, so most windows are flat and can't
distinguish methods — but transitional windows (where temperature ticks up) are decisive.

**Result:** block **mean** matches to ~3e-4 °C; decimation is off by up to 0.03 °C.
**ESS-DIVE temperature = 10-second block mean.**

In [27]:
essd_temp = essd['temperature'].sel(
    time=slice("2023-08-30T12:00:00", "2023-08-30T12:10:00")
).to_series()
essd_temp.index = essd_temp.index.tz_localize("UTC")   # tz-naive -> UTC to match native

compare = pd.DataFrame({
    'essdive':      essd_temp,
    'native_mean':  mean_10s,
    'native_first': first_10s,
}).dropna()   # keep only overlapping 10-s bins

print("rows compared:", len(compare))
print(compare.head(20))

# Max absolute difference is more informative than a bool: mean ~= 0, first is larger.
print("\nmax |essdive - native_mean| :", (compare['essdive'] - compare['native_mean']).abs().max())
print("max |essdive - native_first|:", (compare['essdive'] - compare['native_first']).abs().max())
print("mean  matches:", np.allclose(compare['essdive'], compare['native_mean'],  atol=0.02))
print("first matches:", np.allclose(compare['essdive'], compare['native_first'], atol=0.02))


rows compared: 60
                           essdive  native_mean  native_first
2023-08-30 12:00:00+00:00    13.70    13.700000          13.7
2023-08-30 12:00:10+00:00    13.70    13.700000          13.7
2023-08-30 12:00:20+00:00    13.70    13.700000          13.7
2023-08-30 12:00:30+00:00    13.70    13.700000          13.7
2023-08-30 12:00:40+00:00    13.70    13.700000          13.7
2023-08-30 12:00:50+00:00    13.70    13.700000          13.7
2023-08-30 12:01:00+00:00    13.70    13.700000          13.7
2023-08-30 12:01:10+00:00    13.70    13.700000          13.7
2023-08-30 12:01:20+00:00    13.70    13.700000          13.7
2023-08-30 12:01:30+00:00    13.70    13.700000          13.7
2023-08-30 12:01:40+00:00    13.70    13.700000          13.7
2023-08-30 12:01:50+00:00    13.70    13.700000          13.7
2023-08-30 12:02:00+00:00    13.73    13.730328          13.7
2023-08-30 12:02:10+00:00    13.80    13.800000          13.8
2023-08-30 12:02:20+00:00    13.80    13.800000     

## 6. Wind: the discriminating test

Wind varies within each 10-second window, so aggregation methods diverge much more clearly
than they do for temperature.

**Result:** `wind_max_10s` matches the native 10-second block maximum of
`wxt.wind.speed` exactly. This is an important alignment check: it shows that the native
Sage wind-speed stream, the ESS-DIVE timestamps, and the 10-second window boundaries are
being compared correctly.

`wind_mean_10s`, however, does **not** match the direct 10-second scalar mean of native
`wxt.wind.speed`. Because vector averaging can produce a mean wind speed smaller than
the scalar mean when direction varies, the next section tests whether ESS-DIVE used a
vector wind calculation.

In [28]:
dfw = sage_data_client.query(
    start="2023-08-30T12:00:00Z", end="2023-08-30T12:10:00Z",
    filter={"name": "wxt.wind.speed", "vsn": "W08D"},
)
natw = dfw[['timestamp', 'value']].copy()
natw['timestamp'] = pd.to_datetime(natw['timestamp'], utc=True)
natw = natw.set_index('timestamp')['value']

w_mean = natw.resample('10s').mean()   # scalar mean of speed
w_max  = natw.resample('10s').max()    # block max of speed

essd_wmean = essd['wind_mean_10s'].sel(time=slice("2023-08-30T12:00:00", "2023-08-30T12:10:00")).to_series()
essd_wmax  = essd['wind_max_10s'].sel(time=slice("2023-08-30T12:00:00", "2023-08-30T12:10:00")).to_series()
essd_wmean.index = essd_wmean.index.tz_localize("UTC")
essd_wmax.index  = essd_wmax.index.tz_localize("UTC")

wc = pd.DataFrame({'essd_mean': essd_wmean, 'nat_mean': w_mean,
                   'essd_max':  essd_wmax,  'nat_max':  w_max}).dropna()
print(wc.head(15))
print("\nmax |essd_mean - nat_mean|:", (wc['essd_mean'] - wc['nat_mean']).abs().max())  # large -> NOT scalar mean
print("max |essd_max  - nat_max| :", (wc['essd_max']  - wc['nat_max']).abs().max())     # 0.0 -> exact block max


                           essd_mean  nat_mean  essd_max  nat_max
2023-08-30 12:00:00+00:00   0.800000  0.730252       1.1      1.1
2023-08-30 12:00:10+00:00   0.458333  0.934711       1.8      1.8
2023-08-30 12:00:20+00:00   1.866667  1.900000       2.1      2.1
2023-08-30 12:00:30+00:00   1.733333  1.617213       1.8      1.8
2023-08-30 12:00:40+00:00   1.508333  1.733607       2.0      2.0
2023-08-30 12:00:50+00:00   1.408333  2.024590       2.7      2.7
2023-08-30 12:01:00+00:00   2.483333  2.077686       2.5      2.5
2023-08-30 12:01:10+00:00   2.425000  2.629508       2.9      2.9
2023-08-30 12:01:20+00:00   2.191667  1.716529       2.2      2.2
2023-08-30 12:01:30+00:00   0.972727  0.568966       1.1      1.1
2023-08-30 12:01:40+00:00   0.472727  0.771304       1.3      1.3
2023-08-30 12:01:50+00:00   1.300000  1.393388       1.5      1.5
2023-08-30 12:02:00+00:00   1.058333  0.904918       1.3      1.3
2023-08-30 12:02:10+00:00   1.200000  1.402459       1.7      1.7
2023-08-30

In [29]:
# Candidate: two-stage scalar averaging
# First average the high-rate stream to 1-second bins,
# then average those 1-second means into 10-second bins.
one_sec_mean = natw.resample("1s").mean()
two_stage_10s = one_sec_mean.resample("10s").mean()

d = pd.DataFrame({
    "essd_mean": essd_wmean,
    "direct_10s_mean": w_mean,
    "two_stage_1s_to_10s": two_stage_10s,
}).dropna()

print(d.head(15))
print("\nmax |essd - direct 10s mean|:",
      (d["essd_mean"] - d["direct_10s_mean"]).abs().max())
print("max |essd - two-stage mean|:",
      (d["essd_mean"] - d["two_stage_1s_to_10s"]).abs().max())

                           essd_mean  direct_10s_mean  two_stage_1s_to_10s
2023-08-30 12:00:00+00:00   0.800000         0.730252             0.731346
2023-08-30 12:00:10+00:00   0.458333         0.934711             0.931667
2023-08-30 12:00:20+00:00   1.866667         1.900000             1.900000
2023-08-30 12:00:30+00:00   1.733333         1.617213             1.617564
2023-08-30 12:00:40+00:00   1.508333         1.733607             1.732372
2023-08-30 12:00:50+00:00   1.408333         2.024590             2.024679
2023-08-30 12:01:00+00:00   2.483333         2.077686             2.077855
2023-08-30 12:01:10+00:00   2.425000         2.629508             2.629167
2023-08-30 12:01:20+00:00   2.191667         1.716529             1.715152
2023-08-30 12:01:30+00:00   0.972727         0.568966             0.566409
2023-08-30 12:01:40+00:00   0.472727         0.771304             0.775576
2023-08-30 12:01:50+00:00   1.300000         1.393388             1.394231
2023-08-30 12:02:00+00:00

**Two-stage scalar averaging does not explain the mismatch.** Averaging the native
high-rate speed to 1-second means and then averaging those 1-second means to 10 seconds
gives almost the same discrepancy as the direct 10-second scalar mean. The mismatch is
therefore not mainly caused by unequal numbers of native samples within each second.

## 7. Did they compute a *vector* wind mean?

A physically defensible way to average wind is to decompose each speed/direction sample
into eastward/northward components, average the components, and then take the magnitude
of the mean vector. This vector-resultant speed is less than or equal to the scalar mean
when wind direction varies within the averaging window.

**Result:** the vector-resultant wind speed also does **not** match ESS-DIVE
`wind_mean_10s`. For this test window, `wind_mean_10s` is not reproduced by either a
direct scalar mean or a vector mean computed from the co-timestamped native
`wxt.wind.speed` and `wxt.wind.direction` samples.

In [30]:
# Pull native wind DIRECTION for the same window.
dfd = sage_data_client.query(
    start="2023-08-30T12:00:00Z", end="2023-08-30T12:10:00Z",
    filter={"name": "wxt.wind.direction", "vsn": "W08D"},
)

spd = natw.copy()  # native wind speed (from cell 6), tz-aware index
drc = dfd[['timestamp', 'value']].copy()
drc['timestamp'] = pd.to_datetime(drc['timestamp'], utc=True)
drc = drc.set_index('timestamp')['value']

print("speed samples:", len(spd), " direction samples:", len(drc))

speed samples: 7257  direction samples: 7257


In [31]:
# Align speed & direction on their shared timestamps before decomposing.
w = pd.DataFrame({'spd': spd, 'drc': drc}).dropna()   # inner-join on index

# Decompose to u/v (meteorological: direction = FROM which the wind blows).
u = -w['spd'] * np.sin(np.deg2rad(w['drc']))
v = -w['spd'] * np.cos(np.deg2rad(w['drc']))

u10 = u.resample('10s').mean()
v10 = v.resample('10s').mean()
vec_mean_speed = np.sqrt(u10**2 + v10**2)   # magnitude of the mean vector

cmp2 = pd.DataFrame({
    'essd_mean':   essd_wmean,
    'scalar_mean': w_mean,
    'vector_mean': vec_mean_speed,
}).dropna()

print(cmp2.head(15))
print("\nmax |essd - scalar|:", (cmp2['essd_mean'] - cmp2['scalar_mean']).abs().max())
print("max |essd - vector|:", (cmp2['essd_mean'] - cmp2['vector_mean']).abs().max())


                           essd_mean  scalar_mean  vector_mean
2023-08-30 12:00:00+00:00   0.800000     0.730252     0.644029
2023-08-30 12:00:10+00:00   0.458333     0.934711     0.922292
2023-08-30 12:00:20+00:00   1.866667     1.900000     1.888185
2023-08-30 12:00:30+00:00   1.733333     1.617213     1.615137
2023-08-30 12:00:40+00:00   1.508333     1.733607     1.591624
2023-08-30 12:00:50+00:00   1.408333     2.024590     1.756083
2023-08-30 12:01:00+00:00   2.483333     2.077686     1.930756
2023-08-30 12:01:10+00:00   2.425000     2.629508     2.615682
2023-08-30 12:01:20+00:00   2.191667     1.716529     1.702633
2023-08-30 12:01:30+00:00   0.972727     0.568966     0.472321
2023-08-30 12:01:40+00:00   0.472727     0.771304     0.755659
2023-08-30 12:01:50+00:00   1.300000     1.393388     1.335978
2023-08-30 12:02:00+00:00   1.058333     0.904918     0.848731
2023-08-30 12:02:10+00:00   1.200000     1.402459     1.385522
2023-08-30 12:02:20+00:00   1.425000     1.341803     1

## 8. Testing ordinary explanations for the wind-mean mismatch

The vector mean mismatch is unexpected, so the following cells test ordinary explanations:
sample-count effects, missing values, speed/direction alignment, alternate wind channels,
bin-boundary conventions, sub-bin phase offsets, metadata mixing, and one-bin timestamp
shifts.

**(a) Sample count per bin** → ~120 per bin = **~12 Hz** (not nominal 10 Hz;
corroborated by earlier analysis). Not the cause, but confirms the true rate.

In [32]:
# how many native samples fall in each of our 10-s bins?
counts = natw.resample('10s').count()
print(counts.head(15))
# ~100 expected for 10 Hz over 10 s; if far from 100, missing-value or rate issue

timestamp
2023-08-30 12:00:00+00:00    119
2023-08-30 12:00:10+00:00    121
2023-08-30 12:00:20+00:00    122
2023-08-30 12:00:30+00:00    122
2023-08-30 12:00:40+00:00    122
2023-08-30 12:00:50+00:00    122
2023-08-30 12:01:00+00:00    121
2023-08-30 12:01:10+00:00    122
2023-08-30 12:01:20+00:00    121
2023-08-30 12:01:30+00:00    116
2023-08-30 12:01:40+00:00    115
2023-08-30 12:01:50+00:00    121
2023-08-30 12:02:00+00:00    122
2023-08-30 12:02:10+00:00    122
2023-08-30 12:02:20+00:00    122
Freq: 10s, Name: value, dtype: int64


**(b) Missing-value contamination** → **zero** `-9999.9` fills, clean range 0.2-3.4
m/s. Ruled out.

In [33]:
print("any -9999.9 in native speed?", (natw <= -9999).sum())
print("native speed range:", natw.min(), natw.max())

any -9999.9 in native speed? 0
native speed range: 0.2 3.4


**(c) Speed/direction alignment** → **7257/7257** paired; the vector mean used the
full dataset. Ruled out.

In [34]:
print("speed samples:", len(spd))
print("direction samples:", len(drc))
w_join = pd.DataFrame({'spd': spd, 'drc': drc}).dropna()
print("after inner-join:", len(w_join))

speed samples: 7257
direction samples: 7257
after inner-join: 7257


**(d) A different wind channel** → the native stream has exactly one wind-speed and
one direction channel. No alternate quantity to average. Ruled out.

In [35]:
# what wind-related channels does the native stream have?
dfall = sage_data_client.query(
    start="2023-08-30T12:00:00Z", end="2023-08-30T12:01:00Z",
    filter={"name": "wxt.*", "vsn": "W08D"},
)
print(sorted(dfall['name'].unique()))

['wxt.env.humidity', 'wxt.env.pressure', 'wxt.env.temp', 'wxt.heater.temp', 'wxt.heater.volt', 'wxt.rain.accumulation', 'wxt.wind.direction', 'wxt.wind.speed']


**(e) Bin-boundary convention** → all four label/closed combinations tested; best is
0.694, others worse (to 1.47). No convention matches. Ruled out.

In [36]:
for lbl in ['left','right']:
    for cl in ['left','right']:
        m = natw.resample('10s', label=lbl, closed=cl).mean()
        d = pd.DataFrame({'e':essd_wmean,'n':m}).dropna()
        print(f"label={lbl} closed={cl}  max|diff|={(d['e']-d['n']).abs().max():.4f}")

label=left closed=left  max|diff|=0.6944
label=left closed=right  max|diff|=0.6944
label=right closed=left  max|diff|=1.4658
label=right closed=right  max|diff|=1.4658


**(f) Sub-bin phase offset** → shifting native bins 0-9 s only makes it monotonically
worse; offset 0 is optimal. No phase offset. Ruled out.

In [37]:
import pandas as pd
for shift_s in range(0, 10):
    shifted = natw.copy()
    shifted.index = shifted.index - pd.Timedelta(seconds=shift_s)
    m = shifted.resample('10s').mean()
    d = pd.DataFrame({'e': essd_wmean, 'n': m}).dropna()
    print(f"shift={shift_s}s  max|diff|={(d['e']-d['n']).abs().max():.4f}")

shift=0s  max|diff|=0.6944
shift=1s  max|diff|=0.8028
shift=2s  max|diff|=0.8936
shift=3s  max|diff|=0.9919
shift=4s  max|diff|=1.0712
shift=5s  max|diff|=1.1999
shift=6s  max|diff|=1.3332
shift=7s  max|diff|=1.4807
shift=8s  max|diff|=1.6367
shift=9s  max|diff|=1.7725


**(g) Mixed metadata path** → the native `wxt.wind.speed` rows all come from one VSN,
one sensor, one task, one plugin version, one host, and one unit string. The mismatch is
therefore unlikely to be caused by accidentally combining multiple Sage data paths.

In [38]:
for col in [c for c in dfw.columns if c.startswith("meta.")]:
    print("\n", col)
    print(dfw[col].value_counts(dropna=False).head(10))


 meta.host
meta.host
000048b02d3ae277.ws-nxcore    7257
Name: count, dtype: int64

 meta.missing
meta.missing
-9999.9    7257
Name: count, dtype: int64

 meta.node
meta.node
000048b02d3ae277    7257
Name: count, dtype: int64

 meta.plugin
meta.plugin
registry.sagecontinuum.org/jrobrien/waggle-wxt536:0.23.5.08    7257
Name: count, dtype: int64

 meta.sensor
meta.sensor
vaisala-wxt536    7257
Name: count, dtype: int64

 meta.task
meta.task
waggle-wxt536    7257
Name: count, dtype: int64

 meta.units
meta.units
meters per second    7257
Name: count, dtype: int64

 meta.vsn
meta.vsn
W08D    7257
Name: count, dtype: int64

 meta.zone
meta.zone
core    7257
Name: count, dtype: int64


**(h) One-bin timestamp shift** → shifting the direct and two-stage 10-second means by
±1 to ±3 bins makes the mean absolute error worse. The best alignment is the unshifted
comparison, so the mismatch is not a simple one-bin labeling or indexing error.

In [39]:
# Test whether ESS-DIVE wind_mean_10s is shifted by one or more 10-second bins

cmp = pd.DataFrame({
    "essd_mean": essd_wmean,
    "direct_10s_mean": w_mean,
    "two_stage_1s_to_10s": two_stage_10s,
}).dropna()

for shift in range(-3, 4):
    shifted = cmp["direct_10s_mean"].shift(shift)
    err = (cmp["essd_mean"] - shifted).abs()
    print(
        f"direct mean shift {shift:+d} bins: "
        f"mean abs err = {err.mean():.4f}, max abs err = {err.max():.4f}"
    )

print()

for shift in range(-3, 4):
    shifted = cmp["two_stage_1s_to_10s"].shift(shift)
    err = (cmp["essd_mean"] - shifted).abs()
    print(
        f"two-stage mean shift {shift:+d} bins: "
        f"mean abs err = {err.mean():.4f}, max abs err = {err.max():.4f}"
    )

direct mean shift -3 bins: mean abs err = 0.5674, max abs err = 1.9144
direct mean shift -2 bins: mean abs err = 0.6308, max abs err = 2.2874
direct mean shift -1 bins: mean abs err = 0.5129, max abs err = 1.8542
direct mean shift +0 bins: mean abs err = 0.2425, max abs err = 0.6944
direct mean shift +1 bins: mean abs err = 0.3211, max abs err = 1.4658
direct mean shift +2 bins: mean abs err = 0.5467, max abs err = 2.1825
direct mean shift +3 bins: mean abs err = 0.5993, max abs err = 2.1568

two-stage mean shift -3 bins: mean abs err = 0.5675, max abs err = 1.9169
two-stage mean shift -2 bins: mean abs err = 0.6309, max abs err = 2.2877
two-stage mean shift -1 bins: mean abs err = 0.5129, max abs err = 1.8550
two-stage mean shift +0 bins: mean abs err = 0.2425, max abs err = 0.6935
two-stage mean shift +1 bins: mean abs err = 0.3208, max abs err = 1.4642
two-stage mean shift +2 bins: mean abs err = 0.5467, max abs err = 2.1808
two-stage mean shift +3 bins: mean abs err = 0.5993, max a

## 9. Wind-direction cross-check

In [40]:
essd_wdir = essd["wind_dir_10s"].sel(
    time=slice("2023-08-30T12:00:00", "2023-08-30T12:10:00")
).to_series()
essd_wdir.index = essd_wdir.index.tz_localize("UTC")

vec_mean_dir = (np.degrees(np.arctan2(-u10, -v10)) + 360) % 360

def angle_diff(a, b):
    return ((a - b + 180) % 360) - 180

dcmp = pd.DataFrame({
    "essd_wdir": essd_wdir,
    "vec_mean_dir": vec_mean_dir,
}).dropna()

print(dcmp.head(15))
print("max circular diff:",
      np.abs(angle_diff(dcmp["essd_wdir"], dcmp["vec_mean_dir"])).max())

                            essd_wdir  vec_mean_dir
2023-08-30 12:00:00+00:00  303.090909    352.542249
2023-08-30 12:00:10+00:00    6.583333    355.795325
2023-08-30 12:00:20+00:00  337.916667    348.888394
2023-08-30 12:00:30+00:00  352.083333    351.887483
2023-08-30 12:00:40+00:00   10.250000    348.323636
2023-08-30 12:00:50+00:00  291.250000    326.711260
2023-08-30 12:01:00+00:00  331.666667    337.399946
2023-08-30 12:01:10+00:00  340.166667    325.588754
2023-08-30 12:01:20+00:00  329.416667    334.198379
2023-08-30 12:01:30+00:00  154.818182    292.101128
2023-08-30 12:01:40+00:00   14.454545     13.585244
2023-08-30 12:01:50+00:00   21.666667      2.956433
2023-08-30 12:02:00+00:00  346.333333      8.363503
2023-08-30 12:02:10+00:00   31.583333     16.838003
2023-08-30 12:02:20+00:00  347.583333    351.312569
max circular diff: 146.94987848460548


The ESS-DIVE wind direction also does not reproduce the vector-mean direction computed
from the native speed/direction samples. That strengthens the interpretation that
ESS-DIVE `wind_mean_10s` and `wind_dir_10s` may have come from a different wind-processing
path than the raw high-rate speed/direction samples used here.

## 10. Conclusions

### Verified reconstructions

| ESS-DIVE variable | reconstruction | max abs. difference |
|---|---|---|
| `temperature` | 10-second block **mean** of native high-rate `wxt.env.temp` | ~3e-4 °C |
| `wind_max_10s` | 10-second block **max** of native `wxt.wind.speed` | **0.0 (exact)** |

The exact match for `wind_max_10s` is the strongest alignment check in the notebook:
it shows that the native Sage wind-speed channel, ESS-DIVE timestamps, and 10-second
windows are being compared correctly.

### `wind_mean_10s` is not reproduced by the tested native-sample aggregations

| candidate | result |
|---|---|
| direct 10-second scalar mean of speed | does not match |
| two-stage 1-second → 10-second scalar mean | does not match |
| vector-resultant mean from native speed/direction | does not match |
| bin-boundary variants | do not match |
| sub-bin phase offsets | do not match |
| ±1 to ±3 10-second bin shifts | do not match |

The native Sage wind-speed data in this test window is internally uniform: one VSN,
one sensor, one task, one plugin version, and one unit string. The mismatch is therefore
not explained by mixed metadata paths, missing-value contamination, speed/direction
misalignment, alternate wind-speed channels, or simple timestamp labeling.

### Working interpretation

For this NEIU/W08D test window, ESS-DIVE `temperature` is consistent with a 10-second
block mean of the native Sage stream, and `wind_max_10s` is exactly the native 10-second
block maximum. However, `wind_mean_10s` is not reproduced by the standard scalar or
vector aggregations tested here.

The most likely explanation is that ESS-DIVE `wind_mean_10s` came from a different
processing path than the raw high-rate `wxt.wind.speed` samples used to reproduce
`wind_max_10s` — possibly a firmware- or plugin-derived wind statistic, or an
intermediate product used by the ESS-DIVE processing workflow. Confirming that requires
the processing code or instrument/plugin configuration.

### Documentation discrepancies

1. The ESS-DIVE description implies thermodynamic channels are 1 Hz, but the native
   Sage stream in this test window is about 12 Hz.
2. The published 10-second wind product is not sufficient by itself to support turbulence
   analysis unless the wind-processing pathway is documented and validated.
3. ESS-DIVE wind-speed metadata should be treated cautiously; project notes already flag
   wind-speed unit defects in the published NetCDF files. 
### Reproducibility note

These results come from one 10-minute window: 2023-08-30 12:00–12:10 UTC at NEIU/W08D.
The temperature and wind-maximum reconstructions are strong for this window. The
`wind_mean_10s` conclusion should be repeated over additional windows and sites before
being treated as a network-wide result.